Perhaps, one of the simplest and most basic operations in Google Earth Engine is to retrieve information for a single point. Typical examples include elevation, mean annual temperature, and soil properties. In other words, this type of operation is mostly intended for retrieving information that does not change with time. 

In [1]:
# Import modules
import ee
from datetime import datetime, timezone
from pprint import pprint

In [2]:
# Authenticate
ee.Authenticate()

# Initialize the library.
ee.Initialize()


## Example 1: Elevation

In the following example we will retrieve the elevation for a specific point on Earth.

**Product**: USGS/SRTMGL1_003

In [3]:
# Define geographic coordinates
lat = 39.186512 # This is y
lon = -96.576844 # This is x

# Convert coordinates into a Point geometry following the x,y notation
point = ee.Geometry.Point([lon, lat])

# Explore point geometry
point.getInfo()


{'type': 'Point', 'coordinates': [-96.576844, 39.186512]}

In [4]:
# Load digital elevation model (DEM) from Shuttle Radar Topography Mission (SRTM) 
dem = ee.Image('USGS/SRTMGL1_003')


In [5]:
# Obtain some information about the DEM layer (output is long!)
pprint(dem.getInfo())


{'bands': [{'crs': 'EPSG:4326',
            'crs_transform': [0.0002777777777777778,
                              0,
                              -180.0001388888889,
                              0,
                              -0.0002777777777777778,
                              60.00013888888889],
            'data_type': {'max': 32767,
                          'min': -32768,
                          'precision': 'int',
                          'type': 'PixelType'},
            'dimensions': [1296001, 417601],
            'id': 'elevation'}],
 'id': 'USGS/SRTMGL1_003',
 'properties': {'date_range': [950227200000, 951177600000],
                'description': '<p>The Shuttle Radar Topography Mission (SRTM, '
                               'see <a '
                               'href="https://onlinelibrary.wiley.com/doi/10.1029/2005RG000183/full">Farr\n'
                               'et al. 2007</a>)\n'
                               'digital elevation data is an internation

In [6]:
# Retrieve the elevation value: This step will get us close to the answer, but we are not there yet
pprint(dem.sample(point, 1).getInfo())


{'columns': {'elevation': 'Short'},
 'features': [{'geometry': None,
               'id': '0',
               'properties': {'elevation': 317},
               'type': 'Feature'}],
 'properties': {'band_order': ['elevation']},
 'type': 'FeatureCollection'}


In [7]:
# Retrieve the elevation value: This step will get us even closer to the answer, but we are not there yet
dem.sample(point, 1).first().getInfo()


{'type': 'Feature',
 'geometry': None,
 'id': '0',
 'properties': {'elevation': 317}}

In [8]:
# Retrieve the elevation value: This step will get us the correct value
elev = dem.sample(point, 1).first().getNumber('elevation').getInfo()
print(f'Elevation: {elev} m')


Elevation: 317 m


## Example 2: Weather variables

Obtain the long-term mean annual air temperature and precipitation for a specific location

**Product**: For more information about bands and units visit: [WorldClim BIO](https://developers.google.com/earth-engine/datasets/catalog/WORLDCLIM_V1_BIO)

In [9]:
# Define geographic coordinates
lat = 39.186512 # This is y
lon = -96.576844 # This is x

# Convert coordinates into a Point geometry following the x,y notation
point = ee.Geometry.Point(lon, lat)


In [10]:
# Load WorldClim BIO dataset
dataset = ee.Image('WORLDCLIM/V1/BIO')


In [ ]:
# Remove the comment to access metadata of the product (output is long!)
# dataset.getInfo()

In [11]:
# Get long-term mean annual air temperature
T_mean = dataset.select('bio01').sample(point,1).first().getNumber('bio01').multiply(0.1).getInfo()
print(f'Mean annual air temperature: {round(T_mean,1)} Celsius')


Mean annual air temperature: 12.2 Celsius


In [12]:
# Get long-term annual precipitation
P_annual = dataset.select('bio12').sample(point,1).first().get('bio12').getInfo()
print(f'Mean annual precipitation: {round(P_annual,1)} mm')


Mean annual precipitation: 857 mm


## Example 3: Reference ET

In this example we will retrieve daily values of reference evapotranspiration for a point.

**Product**: For more information visit the description in Google Earth Engine of [GRIDMET](https://developers.google.com/earth-engine/datasets/catalog/IDAHO_EPSCOR_GRIDMET?hl=en)


In [13]:
# Define point
point = ee.Geometry.Point([-96.576844, 39.186512])


In [14]:
# Obtain GRIDMET dataset for a specific period. End date is excluded from the call
start_date = ee.Date('2022-07-01')
end_date = ee.Date('2022-07-02')
dataset = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET').filterDate(start_date, end_date)


In [ ]:
# Remove the comment to see information about the image collection (output is long!)
# pprint(dataset.getInfo())

In [15]:
# Use the get method to retrieve a specific property
dataset.get('description').getInfo()


'<p>The Gridded Surface Meteorological dataset provides\nhigh spatial resolution (~4-km) daily surface fields of temperature,\nprecipitation, winds, humidity and radiation across the contiguous\nUnited States from 1979. The dataset blends the high resolution\nspatial data from PRISM with the high temporal resolution data\nfrom the National Land Data Assimilation System (NLDAS) to produce\nspatially and temporally continuous fields that lend themselves\nto additional land surface modeling.</p><p>This dataset contains provisional products that are replaced with\nupdated versions when the complete source data become available.\nProducts can be distinguished by the value of the &#39;status&#39; property.\nAt first, assets are ingested with status=&#39;early&#39;. After several\ndays, they are replaced by assets with status=&#39;provisional&#39;.\nAfter about 2 months, they are replaced by the final assets with\nstatus=&#39;permanent&#39;.</p><p><b>Provider: <a href="http://www.climatologyl

In [16]:
# Information about first image within the collection (output is long!)
dataset.first().getInfo()


{'type': 'Image',
 'bands': [{'id': 'pr',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [1386, 585],
   'crs': 'GEOGCS["unknown", \n  DATUM["unknown", \n    SPHEROID["Spheroid", 6378137.0, 298.257223563]], \n  PRIMEM["Greenwich", 0.0], \n  UNIT["degree", 0.017453292519943295], \n  AXIS["Longitude", EAST], \n  AXIS["Latitude", NORTH]]',
   'crs_transform': [0.041666666666666664,
    0,
    -124.78749996666667,
    0,
    -0.041666666666666664,
    49.42083333333334]},
  {'id': 'rmax',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [1386, 585],
   'crs': 'GEOGCS["unknown", \n  DATUM["unknown", \n    SPHEROID["Spheroid", 6378137.0, 298.257223563]], \n  PRIMEM["Greenwich", 0.0], \n  UNIT["degree", 0.017453292519943295], \n  AXIS["Longitude", EAST], \n  AXIS["Latitude", NORTH]]',
   'crs_transform': [0.041666666666666664,
    0,
    -124.78749996666667,
    0,
    -0.041666666666666664,
    49.42083333333334]},
  {'id': 'rmin',


In [17]:
# Information about feature collection
dataset.first().sample(point,1).getInfo() # Sample at 1 meter resolution


{'type': 'FeatureCollection',
 'columns': {'bi': 'Float',
  'erc': 'Float',
  'eto': 'Float',
  'etr': 'Float',
  'fm100': 'Float',
  'fm1000': 'Float',
  'pr': 'Float',
  'rmax': 'Float',
  'rmin': 'Float',
  'sph': 'Float',
  'srad': 'Float',
  'th': 'Float',
  'tmmn': 'Float',
  'tmmx': 'Float',
  'vpd': 'Float',
  'vs': 'Float'},
 'properties': {'band_order': ['pr',
   'rmax',
   'rmin',
   'sph',
   'srad',
   'th',
   'tmmn',
   'tmmx',
   'vs',
   'erc',
   'eto',
   'bi',
   'fm100',
   'fm1000',
   'etr',
   'vpd']},
 'features': [{'type': 'Feature',
   'geometry': None,
   'id': '0',
   'properties': {'bi': 0,
    'erc': 20,
    'eto': 4.400000095367432,
    'etr': 5.099999904632568,
    'fm100': 15.199999809265137,
    'fm1000': 18,
    'pr': 43,
    'rmax': 92.5,
    'rmin': 62.599998474121094,
    'sph': 0.014340000227093697,
    'srad': 249.39999389648438,
    'th': 114,
    'tmmn': 293.3999938964844,
    'tmmx': 299.5,
    'vpd': 0.6399999856948853,
    'vs': 2.799999952

In [18]:
# Information about feature
eto = dataset.first().sample(point,1).first().getNumber('eto').getInfo()
print(f'The grass reference ET is {eto} mm')

The grass reference ET is 4.400000095367432 mm


Alternative solution: Access the image directly rather than the collection.

```python
ee.Image('IDAHO_EPSCOR/GRIDMET/20220701').sample(point,1).first().getNumber('eto').getInfo()
```

#### Get dataset timestamps

In [19]:
# Obtain the START time of the dataset to check that it matches our request.
# Response is in milliseconds since 1-Jan-1970
dataset.select('eto').first().getNumber('system:time_start').getInfo()


1656655200000

In [20]:
# Obtain the END time of the dataset to check that it matches our request.
# Response is in milliseconds since 1-Jan-1970
dataset.select('eto').first().getNumber('system:time_end').getInfo()


1656741600000

In [21]:
# Find the datetime of the serial date numbers
# Input in "fromtimestamp()" has to be in seconds, so we divide by 1000
# GEE timestamps are in UTC, so we also specify the time zone
print('Start time:', datetime.fromtimestamp(1656655200000/1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S.%f'))
print('End time:', datetime.fromtimestamp(1656741600000/1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S.%f'))


Start time: 2022-07-01 06:00:00.000000
End time: 2022-07-02 06:00:00.000000


## Example 4: Retrieve soil properties for a given location

Product: SoilGrids

Source: https://samapriya.github.io/awesome-gee-community-datasets/projects/isric/

In [22]:
# Load the SoilGrids dataset from GEE
soil_grids = ee.Image("projects/soilgrids-isric/sand_mean")


In [23]:
# Define a point geometry
lat = 37.839154
lon = -99.101594
point = ee.Geometry.Point(lon,lat)

# SoilGrids reports sand content in g/kg, so we multiply by 0.1 to obtain percent
sand = soil_grids.sample(point,250).first().getNumber('sand_0-5cm_mean').multiply(0.1).getInfo()


In [24]:
print(f'The percentage of sand at ({lat},{lon}) is: {round(sand)} %')

The percentage of sand at (37.839154,-99.101594) is: 53 %
